In [1]:
# =========================
# Load data and show head
# =========================

import os
import random
import pandas as pd

TRAIN_CSV = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"
MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

# Optional: if you ever attach a separate tokenizer artifact dataset, set it here.
# Leave as None to use the tokenizer from MODEL_PATH.
LOCAL_TOKENIZER_PATH = None

OUTPUT_DIR = "/kaggle/working/"

RANDOM_SEED = 42
# Use more rows — 96 GB VRAM on RTX PRO 6000 Blackwell handles it;
# higher data volume dramatically improves math reasoning accuracy.
TRAIN_ROWS_LIMIT = 7500
VAL_FRACTION = 0.03

random.seed(RANDOM_SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(TRAIN_CSV)
print(df.head())

if TRAIN_ROWS_LIMIT is not None:
    df = df.iloc[:TRAIN_ROWS_LIMIT].copy()

df = df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

val_size = max(1, int(len(df) * VAL_FRACTION))
val_df = df.iloc[:val_size].copy()
train_df = df.iloc[val_size:].copy()

print(f"train={len(train_df)} val={len(val_df)}")


         id                                             prompt  \
0  00066667  In Alice's Wonderland, a secret bit manipulati...   
1  000b53cf  In Alice's Wonderland, a secret bit manipulati...   
2  00189f6a  In Alice's Wonderland, secret encryption rules...   
3  001b24c4  In Alice's Wonderland, numbers are secretly co...   
4  001c63cb  In Alice's Wonderland, secret encryption rules...   

                  answer  
0               10010111  
1               01000011  
2      cat imagines book  
3                XXXVIII  
4  wizard creates secret  
train=7275 val=225


In [2]:
import sys
import os
import shutil
import stat

# ptxas-blackwell fix (RTX PRO 6000 Blackwell requires updated ptxas)
src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"]           = dst
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst

    try:
        import triton.backends.nvidia.compiler as nv_compiler
        def _safe_get_ptxas_version(*args, **kwargs):
            return (12, 9, 0)
        nv_compiler.get_ptxas_version = _safe_get_ptxas_version
        print("Triton ptxas fix applied ✓")
    except Exception as e:
        print(f"Triton ptxas partial fix (non-fatal): {e}")
else:
    print("ptxas-blackwell not found — continuing without fix")


# =========================
# Create adapter — RTX PRO 6000 Blackwell (96 GB VRAM), no internet
# =========================

import site
import importlib.util
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

# Make local NVIDIA utility packages visible
site.addsitedir("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script")
site.addsitedir("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU is required.")

print("gpu:", torch.cuda.get_device_name(0))
print("vram_gb:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
print("cutlass spec:", importlib.util.find_spec("cutlass"))
print("mamba_ssm spec:", importlib.util.find_spec("mamba_ssm"))
print("sentencepiece spec:", importlib.util.find_spec("sentencepiece"))

if importlib.util.find_spec("cutlass") is None:
    raise RuntimeError(
        "Local CUTLASS Python package is not visible. "
        "The Kaggle NVIDIA utility path was added, but 'cutlass' still was not found."
    )

if importlib.util.find_spec("mamba_ssm") is None:
    raise RuntimeError(
        "Local mamba_ssm package is not visible. "
        "The Kaggle NVIDIA utility path was added, but 'mamba_ssm' still was not found."
    )

# ---- config ----

# Nemotron-3-Nano-30B is a Mamba-2/MoE/Attention hybrid (NemotronH architecture):
#   52 layers total:
#     - 23 Mamba-2 SSM layers  → target: in_proj, out_proj
#     - 23 MoE layers          → target: shared_experts.up_proj / down_proj
#       (targeting all 128 routed experts per layer is memory-prohibitive)
#     - 6 Attention layers     → target: q_proj, k_proj, v_proj, o_proj
#
# The metric runs vLLM with enable_thinking=True, so the model generates
# <think>...</think> reasoning traces before the boxed answer. Training data
# MUST include these traces so the model learns the correct inference format.

# Token budget: 19.5 GiB disk, no internet, 96 GB VRAM.
# Reasoning traces can be hundreds of tokens; answers are short.
# We use a generous token budget to capture full chain-of-thought.
MAX_PROMPT_TOKENS  = 512    # question + system preamble
MAX_THINK_TOKENS   = 1024   # <think>...</think> reasoning trace
MAX_ANSWER_TOKENS  = 128    # final answer after </think>
MAX_TOTAL_TOKENS   = 1664   # sum of above — fits within vLLM max_model_len=8192

# LoRA: r=32 is the max allowed by the metric (max_lora_rank=32).
# use_rslora normalises by sqrt(r), which stabilises training at higher ranks.
# alpha = 2×r → effective scale = 1.0, a standard and safe choice.
LORA_RANK    = 32
LORA_ALPHA   = 64   # 2 × LORA_RANK
LORA_DROPOUT = 0.05

# With 96 GB VRAM we can push batch size up significantly.
# Effective batch = PER_DEVICE_BATCH_SIZE × GRAD_ACCUM_STEPS = 4 × 8 = 32
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS      = 8
# 2 epochs over ~7000 examples gives robust coverage without overfitting.
NUM_EPOCHS     = 2
LEARNING_RATE  = 2e-4
WEIGHT_DECAY   = 0.01
WARMUP_RATIO   = 0.05
LOGGING_STEPS  = 20

# System prompt mirrors how the metric runs the model at inference time.
SYSTEM_PROMPT = (
    "You are a helpful mathematical reasoning assistant. "
    "Think step by step. Place your final answer inside \\boxed{}."
)

def load_tokenizer_offline(model_path: str, tokenizer_fallback_path: str | None = None):
    candidates = []
    if tokenizer_fallback_path and os.path.exists(tokenizer_fallback_path):
        candidates.append(tokenizer_fallback_path)
    candidates.append(model_path)

    errors = []
    for candidate in candidates:
        try:
            tok = AutoTokenizer.from_pretrained(
                candidate,
                trust_remote_code=True,
                use_fast=True,
                local_files_only=True,
            )
            print(f"Loaded FAST tokenizer from: {candidate}")
            return tok
        except Exception as e:
            errors.append(f"FAST tokenizer failed at {candidate}: {repr(e)}")

        if importlib.util.find_spec("sentencepiece") is not None:
            try:
                tok = AutoTokenizer.from_pretrained(
                    candidate,
                    trust_remote_code=True,
                    use_fast=False,
                    local_files_only=True,
                )
                print(f"Loaded SLOW tokenizer from: {candidate}")
                return tok
            except Exception as e:
                errors.append(f"SLOW tokenizer failed at {candidate}: {repr(e)}")

    msg = "\n".join(errors)
    raise RuntimeError(
        "Offline tokenizer loading failed.\n"
        "Because this RTX session has no internet, do not try pip installs here.\n"
        "Either:\n"
        "1) make sure the tokenizer files inside MODEL_PATH are sufficient for fast loading, or\n"
        "2) attach a separate tokenizer artifact dataset and point LOCAL_TOKENIZER_PATH to it.\n\n"
        f"Details:\n{msg}"
    )

tokenizer = load_tokenizer_offline(MODEL_PATH, LOCAL_TOKENIZER_PATH)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def build_chat_text(question: str, think_trace: str, final_answer: str) -> tuple[str, str]:
    """
    Build (prompt_text, completion_text) that mirrors the inference format used
    by the metric (enable_thinking=True via apply_chat_template).

    The model at inference time generates:
        <think>\n{reasoning}\n</think>\n{answer}

    We replicate this exactly so fine-tuning reinforces the same pattern.
    """
    user_content = (
        question
        + "\nPlease put your final answer inside `\\boxed{}`. "
        "For example: `\\boxed{your answer}`"
    )

    # Try to use the model's own chat template (matches inference exactly).
    try:
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "user", "content": user_content}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
    except Exception:
        # Fallback if the template doesn't support enable_thinking kwarg.
        try:
            prompt_text = tokenizer.apply_chat_template(
                [{"role": "user", "content": user_content}],
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception:
            prompt_text = (
                f"<|system|>\n{SYSTEM_PROMPT}\n"
                f"<|user|>\n{user_content}\n"
                f"<|assistant|>\n"
            )

    # Completion: reasoning trace + boxed final answer.
    # If we have a multi-step trace from the dataset use it; otherwise synthesise one.
    clean_answer = str(final_answer).strip()
    if think_trace:
        completion_text = (
            f"<think>\n{think_trace.strip()}\n</think>\n"
            f"The answer is \\boxed{{{clean_answer}}}."
        )
    else:
        # Minimal but correctly formatted thinking block so the model learns
        # the <think>...</think> wrapper even without a full trace.
        completion_text = (
            f"<think>\nLet me work through this step by step.\n"
            f"The answer is {clean_answer}.\n</think>\n"
            f"The answer is \\boxed{{{clean_answer}}}."
        )

    return prompt_text, completion_text


class BoxedReasoningDataset(Dataset):
    def __init__(self, df_in, tokenizer):
        self.df = df_in.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Support optional "solution" / "reasoning" column for chain-of-thought.
        think_trace = ""
        for col in ("solution", "reasoning", "chain_of_thought", "steps"):
            if col in row.index and pd.notna(row.get(col, None)):
                think_trace = str(row[col])
                break

        prompt_text, completion_text = build_chat_text(
            question=str(row["prompt"]),
            think_trace=think_trace,
            final_answer=row["answer"],
        )

        prompt_ids = self.tokenizer(
            prompt_text,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_PROMPT_TOKENS,
        )["input_ids"]

        completion_ids = self.tokenizer(
            completion_text,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_THINK_TOKENS + MAX_ANSWER_TOKENS,
        )["input_ids"]

        eos_id = self.tokenizer.eos_token_id
        input_ids = prompt_ids + completion_ids + ([eos_id] if eos_id is not None else [])
        # Mask prompt tokens from loss — only train on the completion.
        labels = ([-100] * len(prompt_ids)
                  + completion_ids
                  + ([eos_id] if eos_id is not None else []))

        input_ids = input_ids[:MAX_TOTAL_TOKENS]
        labels    = labels[:MAX_TOTAL_TOKENS]
        attention_mask = [1] * len(input_ids)

        return {
            "input_ids":      torch.tensor(input_ids,      dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels":         torch.tensor(labels,         dtype=torch.long),
        }


class DynamicPadCollator:
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, features):
        max_len = max(len(x["input_ids"]) for x in features)

        def pad_tensor(t, pad_value):
            if len(t) == max_len:
                return t
            pad = torch.full((max_len - len(t),), pad_value, dtype=t.dtype)
            return torch.cat([t, pad], dim=0)

        return {
            "input_ids":      torch.stack([pad_tensor(x["input_ids"],      self.pad_token_id) for x in features]),
            "attention_mask": torch.stack([pad_tensor(x["attention_mask"], 0)                 for x in features]),
            "labels":         torch.stack([pad_tensor(x["labels"],         -100)              for x in features]),
        }


# Load full model in BF16 on the RTX PRO 6000 (96 GB VRAM).
# 30B × 2 bytes ≈ 60 GB weights — fits comfortably with headroom for gradients.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    local_files_only=True,
    device_map={"": 0},
)

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

model.gradient_checkpointing_enable()

# ---- LoRA targeting Nemotron-H's three layer types ----
#
# NemotronH architecture breakdown (52 layers):
#   6  Attention layers → q_proj, k_proj, v_proj, o_proj
#   23 Mamba-2 layers   → in_proj, out_proj
#   23 MoE layers       → shared_experts.up_proj, shared_experts.down_proj
#                         (routed experts skipped — 128 × 23 = too many adapters)
#
# r=32 = max_lora_rank allowed by the metric.
# use_rslora=True scales by 1/sqrt(r), keeping effective LR stable at high rank.
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    use_rslora=True,
    target_modules=[
        # Attention layers (6 total in NemotronH)
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        # Mamba-2 SSM layers (23 total) — critical for sequence reasoning
        "in_proj",
        "out_proj",
        # MoE shared expert (processes every token, high impact per parameter)
        "shared_experts.up_proj",
        "shared_experts.down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_dataset = BoxedReasoningDataset(train_df, tokenizer)
val_dataset   = BoxedReasoningDataset(val_df,   tokenizer)
collator      = DynamicPadCollator(tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",          # cosine decay is better for reasoning
    logging_steps=LOGGING_STEPS,
    save_strategy="epoch",
    eval_strategy="no",
    bf16=True,
    fp16=False,
    optim="adamw_torch_fused",           # faster fused implementation
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    save_total_limit=1,
    # gradient checkpointing already enabled above; enable here too for Trainer
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
)

sample = train_dataset[0]
print("sample lengths:", {k: v.shape for k, v in sample.items()})
print(tokenizer.decode(sample["input_ids"][:300], skip_special_tokens=False))

trainer.train()

# Save adapter files, including adapter_config.json
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved files:", os.listdir(OUTPUT_DIR))
print("adapter_config.json exists:", os.path.exists(os.path.join(OUTPUT_DIR, "adapter_config.json")))

Triton ptxas fix applied ✓


/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


torch: 2.12.0.dev20260324+cu128
cuda available: True
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition
vram_gb: 94.97
cutlass spec: ModuleSpec(name='cutlass', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7bd7efff0d70>, origin='/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/cutlass/__init__.py', submodule_search_locations=['/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/cutlass'])
mamba_ssm spec: ModuleSpec(name='mamba_ssm', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7bd7efb15160>, origin='/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/mamba_ssm/__init__.py', submodule_search_locations=['/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/mamba_ssm'])
sentencepiece spec: ModuleSpec(name='sentencepiece', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7bda7710f740>, origin='/usr/local/lib/python3.12/dist-packages/

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded FAST tokenizer from: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 27,711,488 || all params: 31,605,648,832 || trainable%: 0.0877
sample lengths: {'input_ids': torch.Size([219]), 'attention_mask': torch.Size([219]), 'labels': torch.Size([219])}
<|im_start|>system
<|im_end|>
<|im_start|>user
In Alice's Wonderland, secret encryption rules are used on text. Here are some examples:
plq obmwlp feqqj bqrgy -> the bright queen reads
feqqj bqrgy plq lmggqj nzpmzj -> queen reads the hidden potion
feqqj ypegmqy irypcq -> queen studies castle
ombg dzejg plq iebmzey yqibqp -> bird found the curious secret
gbrwzj gbqrhy plq iebmzey hqyyrwq -> dragon dreams the curious message
Now, decrypt the following text: hzeyq ilryqy pbqryebq
Please put your final answer inside `\boxed{}`. For example: `\boxed{your answer}`<|im_end|>
<|im_start|>assistant
<think>
<think>
Let me work through this step by step.
The answer is mouse chases treasure.
</think>
The answer is \boxed{mouse chases treasure}.<|im_end|>


Step,Training Loss
20,9.349406
40,1.314919
60,1.160096
80,1.099530
100,1.016434
120,1.044597
140,0.998665
160,0.937848
180,0.880632
200,0.853547


Saved files: ['__notebook__.ipynb', 'README.md', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'adapter_config.json', 'tokenizer_config.json', 'checkpoint-456']
adapter_config.json exists: True


In [3]:
# ---- Zip only the files the metric needs ----
# Disk budget: 19.5 GiB.
# adapter_model.safetensors at r=32 ≈ 1–2 GB — well within budget.
# Include only the files vLLM + peft need to load the LoRA adapter.
import subprocess

files_to_zip = []
for fname in [
    "adapter_config.json",       # required by metric
    "adapter_model.safetensors", # adapter weights (preferred over .bin)
    "adapter_model.bin",         # fallback if safetensors not produced
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "chat_template.jinja",
]:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        files_to_zip.append(fname)

print("Files to include in zip:", files_to_zip)
zip_cmd = "cd /kaggle/working && zip submission.zip " + " ".join(files_to_zip)
subprocess.run(zip_cmd, shell=True, check=True)

# Verify disk usage stays within 19.5 GiB budget
subprocess.run("du -sh /kaggle/working/", shell=True)
subprocess.run("du -sh /kaggle/working/submission.zip", shell=True)

Files to include in zip: ['adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja']
  adding: adapter_config.json (deflated 57%)
  adding: adapter_model.safetensors (deflated 19%)
  adding: tokenizer.json (deflated 84%)
  adding: tokenizer_config.json (deflated 46%)
  adding: chat_template.jinja (deflated 78%)
490M	/kaggle/working/
89M	/kaggle/working/submission.zip


CompletedProcess(args='du -sh /kaggle/working/submission.zip', returncode=0)